In [0]:
# %sql
# -- one-time fix
# DELETE FROM weather_project.default.pipeline_control
# WHERE table_name = 'weather_project.default.bronze_weather_raw';

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, StringType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:

CATALOG = "weather_project"
SCHEMA  = "default"

#  table names
BRONZE_TABLE     = f"{CATALOG}.{SCHEMA}.bronze_weather_raw"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.silver_weather_quarantine"
STATION_TABLE    = f"{CATALOG}.{SCHEMA}.station_master"
SILVER_TABLE     = f"{CATALOG}.{SCHEMA}.silver_weather_clean"
AUDIT_TABLE      = f"{CATALOG}.{SCHEMA}.weather_change_audit"
CONTROL_TABLE    = f"{CATALOG}.{SCHEMA}.pipeline_control"



print("Silver notebook ready.")
print()
print(f"  Reading from  → {BRONZE_TABLE}")
print(f"  Writing to    → {SILVER_TABLE}")
print(f"  Audit table   → {AUDIT_TABLE}")
print(f"  Control table → {CONTROL_TABLE}")

Silver notebook ready.

  Reading from  → weather_project.default.bronze_weather_raw
  Writing to    → weather_project.default.silver_weather_clean
  Audit table   → weather_project.default.weather_change_audit
  Control table → weather_project.default.pipeline_control


In [0]:
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.gold_provider_quality")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.gold_temperature_change")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.gold_rainfall_streaks")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.gold_heatwave_alerts")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.gold_city_weather_snapshot")

# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.silver_weather_clean")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.silver_weather_quarantine")

# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.bronze_weather_raw")

# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.pipeline_control")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.weather_change_audit")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.station_master")


# try:
#     dbutils.fs.rm(f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints", True)
# except:
#     print("checkpoints folder already deleted")

# try:
#     dbutils.fs.rm(f"/Volumes/{CATALOG}/{SCHEMA}/bronze_schema", True)
# except:
#     print("bronze_schema folder already deleted")

In [0]:
# Create control table
#audit_log
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTROL_TABLE}(
  table_name STRING,
  last_processed_version BIGINT,
  updated_at TIMESTAMP
) USING DELTA
""")

# Create Silver clean table
#name shouuld reflect businnes need
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TABLE}(
  city STRING,
  country STRING,
  event_time TIMESTAMP,
  temperature DOUBLE,
  humidity INT,
  wind_speed DOUBLE,
  pressure DOUBLE,
  weather_condition STRING,
  rainfall DOUBLE,
  uv_index DOUBLE,
  visibility DOUBLE,
  feels_like_temperature DOUBLE,
  station_id STRING,
  station_zone STRING,
  data_provider STRING,
  ingestion_time TIMESTAMP,
  source_file STRING,
  event_date DATE,
  event_hour INT,
  temperature_band STRING,
  weather_severity_score INT
)
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

# Create quarantine table
spark.sql(f"DROP TABLE IF EXISTS {QUARANTINE_TABLE}")

spark.sql(f"""
CREATE TABLE {QUARANTINE_TABLE}(
  city STRING,
  country STRING,
  event_time STRING,
  temperature STRING,
  humidity STRING,
  wind_speed STRING,
  pressure STRING,
  weather_condition STRING,
  condition STRING,
  rainfall STRING,
  uv_index STRING,
  visibility STRING,
  feels_like_temperature STRING,
  station_id STRING,
  data_provider STRING,
  ingestion_time TIMESTAMP,
  source_file STRING,
  quarantine_reason STRING
)
USING DELTA
""")
#bronze===append,no schema defined

DataFrame[]

In [0]:
# Read last processed Bronze version from control table
ctrl_df = (
    spark.table(CONTROL_TABLE)
         .where(F.col("table_name") == BRONZE_TABLE)
         .orderBy(F.col("updated_at").desc())
)

ctrl_row = ctrl_df.first()
start_version = 0 if ctrl_row is None else int(ctrl_row["last_processed_version"]) + 1

print(f"Reading Bronze CDF from startingVersion = {start_version}")

Reading Bronze CDF from startingVersion = 2


In [0]:
from pyspark.sql import functions as F


# last processed version from control
ctrl_row = (
    spark.table(CONTROL_TABLE)
         .where(F.col("table_name") == BRONZE_TABLE)
         .orderBy(F.col("updated_at").desc())
         .first()
)
requested_start = 0 if ctrl_row is None else int(ctrl_row["last_processed_version"]) 

# latest available bronze version
latest_version = (
    spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}")
         .agg(F.max("version").alias("v"))
         .first()["v"]
)
latest_version = int(latest_version)

print(f"requested_start={requested_start}, latest_version={latest_version}")

# guard
if requested_start > latest_version:
    print("No new Bronze versions to process.")
    dbutils.notebook.exit("NO_NEW_DATA")

from pyspark.sql import functions as F


cdf_df = (
    spark.read
         .option("readChangeFeed", "true")
         .option("startingVersion",requested_start )  
         .table(BRONZE_TABLE)
         .where(F.col("_change_type").isin("insert", "update_postimage"))
)

if len(cdf_df.take(1)) == 0:
    print("No CDF rows found from version 0.")
    dbutils.notebook.exit("NO_DATA")

requested_start=1, latest_version=1


In [0]:
station_df = spark.table(STATION_TABLE).select(
    F.col("station_id").cast("string").alias("station_id_master"),
    F.col("station_zone").cast("string").alias("master_zone")   
)

In [0]:
base_df = (
    cdf_df
    .withColumn("event_time_ts", F.expr("try_cast(event_time as timestamp)"))
    .withColumn("temperature_d", F.expr("try_cast(temperature as double)"))
    .withColumn("humidity_i", F.expr("try_cast(humidity as int)"))
    .withColumn("wind_speed_d", F.expr("try_cast(wind_speed as double)"))
    .withColumn("pressure_d", F.expr("try_cast(pressure as double)"))
    .withColumn("rainfall_d", F.expr("try_cast(rainfall as double)"))
    .withColumn("uv_index_d", F.expr("try_cast(uv_index as double)"))
    .withColumn("visibility_d", F.expr("try_cast(visibility as double)"))
    .withColumn("feels_like_temperature_d", F.expr("try_cast(feels_like_temperature as double)"))
    .withColumn("station_id_s", F.col("station_id").cast("string"))
    .withColumn("weather_condition_s", F.coalesce(F.col("weather_condition"), F.col("condition")))
)

In [0]:
joined_df = base_df.join(
    station_df,
    base_df["station_id_s"] == station_df["station_id_master"],
    "left"
)

In [0]:
quarantine_reason_col = (
    F.when(F.col("event_time_ts").isNull(), F.lit("invalid_timestamp"))
     .when((F.col("temperature_d") > 60) | (F.col("temperature_d") < -20), F.lit("out_of_range_temperature"))
     .when((F.col("humidity_i") > 100) | (F.col("humidity_i") < 0), F.lit("out_of_range_humidity"))
     .when(F.col("wind_speed_d") < 0, F.lit("out_of_range_wind_speed"))
     .when(F.col("master_zone").isNull() & F.col("source_file").contains("weather_12"),F.lit("unmappable_station_id"))
     .when(F.col("station_id_s").isNotNull() & F.col("master_zone").isNull(), F.lit("station_not_in_master"))
     .when(F.col("temperature").isNotNull() & F.col("temperature_d").isNull(), F.lit("data_type_conflict"))
     .when(F.col("wind_speed").isNotNull() & F.col("wind_speed_d").isNull(), F.lit("data_type_conflict"))
)

tagged_df = joined_df.withColumn("quarantine_reason", quarantine_reason_col)


quarantine_df = tagged_df.where(F.col("quarantine_reason").isNotNull())
clean_df = tagged_df.where(F.col("quarantine_reason").isNull())


In [0]:
if len(quarantine_df.take(1)) > 0:
    (
        quarantine_df.select(
            F.col("city").cast("string").alias("city"),
            F.col("country").cast("string").alias("country"),
            F.col("event_time").cast("string").alias("event_time"),
            F.col("temperature").cast("string").alias("temperature"),
            F.col("humidity").cast("string").alias("humidity"),
            F.col("wind_speed").cast("string").alias("wind_speed"),
            F.col("pressure").cast("string").alias("pressure"),
            F.col("weather_condition").cast("string").alias("weather_condition"),
            F.col("condition").cast("string").alias("condition"),
            F.col("rainfall").cast("string").alias("rainfall"),
            F.col("uv_index").cast("string").alias("uv_index"),
            F.col("visibility").cast("string").alias("visibility"),
            F.col("feels_like_temperature").cast("string").alias("feels_like_temperature"),
            F.col("station_id").cast("string").alias("station_id"),
            F.col("data_provider").cast("string").alias("data_provider"),
            F.expr("try_cast(ingestion_time as timestamp)").alias("ingestion_time"),
            F.col("source_file").cast("string").alias("source_file"),
            F.col("quarantine_reason").cast("string").alias("quarantine_reason")
        )
        .write.format("delta")
        .mode("overwrite")
        .saveAsTable(QUARANTINE_TABLE)
    )#append the quar
    #dataframe.isEmpty instead of take

In [0]:
silver_ready_df = (
    clean_df.select(
        F.col("city").cast("string").alias("city"),
        F.col("country").cast("string").alias("country"),
        F.col("event_time_ts").alias("event_time"),
        F.col("temperature_d").alias("temperature"),
        F.col("humidity_i").alias("humidity"),
        F.col("wind_speed_d").alias("wind_speed"),
        F.col("pressure_d").alias("pressure"),
        F.col("weather_condition_s").cast("string").alias("weather_condition"),
        F.col("rainfall_d").alias("rainfall"),
        F.col("uv_index_d").alias("uv_index"),
        F.col("visibility_d").alias("visibility"),
        F.col("feels_like_temperature_d").alias("feels_like_temperature"),
        F.col("station_id_s").alias("station_id"),
        F.col("master_zone").cast("string").alias("station_zone"),
        F.col("data_provider").cast("string").alias("data_provider"),
        F.expr("try_cast(ingestion_time as timestamp)").alias("ingestion_time"),
        F.col("source_file").cast("string").alias("source_file")
    )
    .withColumn("event_date", F.to_date("event_time"))
    .withColumn("event_hour", F.hour("event_time"))
    .withColumn(
        "temperature_band",
        F.when(F.col("temperature") < 15, "Cold")
         .when((F.col("temperature") >= 15) & (F.col("temperature") <= 30), "Normal")
         .when((F.col("temperature") > 30) & (F.col("temperature") <= 40), "Hot")
         .otherwise("Extreme")
    )
    .withColumn(
        "weather_severity_score",
        F.lit(0)
        + F.when(F.col("temperature") > 40, 3).otherwise(0)
        + F.when(F.col("rainfall") > 30, 2).otherwise(0)
        + F.when(F.col("wind_speed") > 40, 2).otherwise(0)
    )
)

In [0]:
print(f"CDF rows (Bronze): {cdf_df.count()}")
print(f"Rows after JOIN  : {joined_df.count()}")
print(f"Clean rows       : {clean_df.count()}")
print(f"Quarantine rows  : {quarantine_df.count()}")
print(f"Total after split: {clean_df.count() + quarantine_df.count()}")

CDF rows (Bronze): 1620
Rows after JOIN  : 1620
Clean rows       : 1465
Quarantine rows  : 155
Total after split: 1620


In [0]:
from pyspark.sql.window import Window
#onlu ingstion not source file
w = Window.partitionBy("city", "event_time").orderBy(F.col("ingestion_time").desc(), F.col("source_file").desc())

silver_ready_df = (
    silver_ready_df
    .withColumn("rn", F.row_number().over(w))
    .where("rn = 1")
    .drop("rn")
)

In [0]:
if len(silver_ready_df.take(1)) > 0:
    silver_delta = DeltaTable.forName(spark, SILVER_TABLE)
    silver_delta.alias("t").merge(
        silver_ready_df.alias("s"),
        "t.city = s.city AND t.event_time = s.event_time"
    ).whenMatchedUpdate(
        condition="s.ingestion_time > t.ingestion_time",
        set={c: f"s.{c}" for c in silver_ready_df.columns}
    ).whenNotMatchedInsert(
        values={c: f"s.{c}" for c in silver_ready_df.columns}
    ).execute()
print("Silver transformation completed.")

Silver transformation completed.


In [0]:
# Update pipeline control after successful Silver write
#update innstead of deleete
spark.sql(f"DELETE FROM {CONTROL_TABLE} WHERE table_name = '{BRONZE_TABLE}'")
spark.sql(f"""
INSERT INTO {CONTROL_TABLE}
VALUES ('{BRONZE_TABLE}', {int(latest_version)}, current_timestamp())
""")

print("Silver CDF pipeline completed successfully.")

Silver CDF pipeline completed successfully.


In [0]:
#weather_change_audit (FINAL CLEAN VERSION)

from pyspark.sql import functions as F
from functools import reduce

# Read Silver CDF incrementally
df_silver_cdf = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", requested_start) \
    .table(SILVER_TABLE)

#  Separate pre and post images
df_pre  = df_silver_cdf.filter(F.col("_change_type") == "update_preimage")
df_post = df_silver_cdf.filter(F.col("_change_type") == "update_postimage")

update_count = df_post.count()
print(f"Updated rows found: {update_count}")

#Process only if updates exist
if update_count > 0:#isEmpty()

    cols_to_audit = [
        "temperature", "humidity", "wind_speed",
        "pressure", "rainfall", "weather_condition"
    ]

    audit_rows = []
    #Compare old vs new values column by column
    for c in cols_to_audit:

        df_audit_col = df_pre.alias("pre").join(
            df_post.alias("post"),
            on=["station_id", "event_time"],  
            how="inner"
        ).filter(# Keep only rows where value changed
            F.col(f"pre.{c}").cast("string") != F.col(f"post.{c}").cast("string")
        ).select(
            F.col("post.station_id"),
            F.col("post.city"),
            F.col("post.event_time"),
            F.lit(c).alias("changed_column"),
            F.col(f"pre.{c}").cast("string").alias("old_value"),
            F.col(f"post.{c}").cast("string").alias("new_value"),
            F.col("post._change_type").alias("change_type"),
            F.col("post._commit_timestamp").alias("changed_at"),
            F.col("post.source_file")
        )

        audit_rows.append(df_audit_col)

    # Combine all column changes

    if audit_rows:

        df_audit_final = reduce(
            lambda a, b: a.unionByName(b),
            audit_rows
        )

        #  Create audit table (if not exists)

        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {AUDIT_TABLE}
            (
                station_id     STRING,
                city           STRING,
                event_time     STRING,
                changed_column STRING,
                old_value      STRING,
                new_value      STRING,
                change_type    STRING,
                changed_at     TIMESTAMP,
                source_file    STRING
            )
            USING DELTA
        """)

        #  Write audit data

        df_audit_final.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(AUDIT_TABLE)

        print(f"Audit rows written: {df_audit_final.count()}")
        display(df_audit_final)

    else:
        print("No column changes detected.")


# No updates case

else:

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {AUDIT_TABLE}
        (
            station_id     STRING,
            city           STRING,
            event_time     STRING,
            changed_column STRING,
            old_value      STRING,
            new_value      STRING,
            change_type    STRING,
            changed_at     TIMESTAMP,
            source_file    STRING
        )
        USING DELTA
    """)

    print("No updates yet — audit table created empty.")

Updated rows found: 0
No updates yet — audit table created empty.


In [0]:
print("Silver rows:", spark.table(SILVER_TABLE).count())
print("Quarantine rows:", spark.table(QUARANTINE_TABLE).count())

display(spark.sql(f"SELECT * FROM {CONTROL_TABLE} WHERE table_name = '{BRONZE_TABLE}'"))
display(spark.sql(f"SELECT quarantine_reason, COUNT(*) cnt FROM {QUARANTINE_TABLE} GROUP BY quarantine_reason ORDER BY cnt DESC"))

Silver rows: 1285
Quarantine rows: 155


table_name,last_processed_version,updated_at
weather_project.default.bronze_weather_raw,1,2026-04-16T10:52:26.410Z


quarantine_reason,cnt
unmappable_station_id,72
invalid_timestamp,29
out_of_range_temperature,24
station_not_in_master,19
out_of_range_humidity,11


In [0]:
display(spark.table(SILVER_TABLE))

city,country,event_time,temperature,humidity,wind_speed,pressure,weather_condition,rainfall,uv_index,visibility,feels_like_temperature,station_id,station_zone,data_provider,ingestion_time,source_file,event_date,event_hour,temperature_band,weather_severity_score
Bangalore,India,2026-02-22T00:00:00.000Z,27.5,57,11.9,913.3,Foggy,1.5,null,null,null,STN_BLR_02,South,WeatherPro,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_10.csv,2026-02-22,0,Normal,0
Bangalore,India,2026-02-22T01:00:00.000Z,31.1,64,5.9,913.8,Partly Cloudy,3.2,null,null,null,STN_BLR_01,South,WeatherPro,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_10.csv,2026-02-22,1,Hot,0
Bangalore,India,2026-02-22T02:00:00.000Z,29.8,60,12.1,911.9,Clear,1.2,null,null,null,STN_BLR_02,South,WeatherPro,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_10.csv,2026-02-22,2,Normal,0
Bangalore,India,2026-02-22T03:00:00.000Z,30.9,50,7.2,902.6,Clear,6.3,null,null,null,STN_BLR_01,South,WeatherPro,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_10.csv,2026-02-22,3,Hot,0
Bangalore,India,2026-02-22T04:00:00.000Z,28.0,57,7.0,900.7,Haze,9.1,null,null,null,STN_BLR_01,South,WeatherPro,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_10.csv,2026-02-22,4,Normal,0
Bangalore,India,2026-02-22T05:00:00.000Z,24.9,50,5.5,903.0,Partly Cloudy,8.0,null,null,null,STN_BLR_01,South,WeatherPro,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_10.csv,2026-02-22,5,Normal,0
Bangalore,India,2026-02-22T06:00:00.000Z,27.8,71,16.5,901.0,Foggy,4.3,null,null,null,STN_BLR_01,South,WeatherPro,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_10.csv,2026-02-22,6,Normal,0
Bangalore,India,2026-02-22T07:00:00.000Z,24.2,55,14.3,905.5,Thunderstorm,4.7,null,null,null,STN_BLR_02,South,WeatherPro,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_10.csv,2026-02-22,7,Normal,0
Bangalore,India,2026-02-22T08:00:00.000Z,26.2,61,17.8,914.5,Clear,8.2,null,null,null,STN_BLR_01,South,WeatherPro,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_10.csv,2026-02-22,8,Normal,0
Bangalore,India,2026-02-22T09:00:00.000Z,26.0,50,10.0,911.2,Thunderstorm,8.6,null,null,null,STN_BLR_01,South,WeatherPro,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_10.csv,2026-02-22,9,Normal,0


In [0]:
display(spark.table(QUARANTINE_TABLE))

city,country,event_time,temperature,humidity,wind_speed,pressure,weather_condition,condition,rainfall,uv_index,visibility,feels_like_temperature,station_id,data_provider,ingestion_time,source_file,quarantine_reason
Chennai,India,INVALID_TIMESTAMP,35.3,73.0,8.3,null,Cloudy,null,9.0,null,null,null,STN_CHN_01,WeatherPro,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_08.csv,invalid_timestamp
Hyderabad,India,INVALID_TIMESTAMP,29.5,69.0,9.1,null,Thunderstorm,null,11.2,null,null,null,STN_HYD_01,ClimaData,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_08.csv,invalid_timestamp
Pune,India,INVALID_TIMESTAMP,24.0,74.0,18.4,null,Haze,null,3.0,null,null,null,STN_PNE_02,SkyWatch,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_08.csv,invalid_timestamp
Chennai,India,INVALID_TIMESTAMP,34.1,82.0,12.4,null,Rainy,null,8.8,null,null,null,STN_CHN_01,WeatherPro,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_08.csv,invalid_timestamp
Hyderabad,India,INVALID_TIMESTAMP,26.3,47.0,11.4,null,Thunderstorm,null,10.4,null,null,null,STN_HYD_01,WeatherPro,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_08.csv,invalid_timestamp
Delhi,India,INVALID_TIMESTAMP,38.9,52.0,12.9,null,Rainy,null,1.4,null,null,null,STN_DEL_01,WeatherPro,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_08.csv,invalid_timestamp
Bangalore,India,INVALID_TIMESTAMP,31.7,74.0,14.1,null,Thunderstorm,null,6.4,null,null,null,STN_BLR_01,SkyWatch,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_08.csv,invalid_timestamp
Chennai,India,INVALID_TIMESTAMP,35.8,74.0,12.0,null,Foggy,null,12.2,null,null,null,STN_CHN_01,SkyWatch,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_08.csv,invalid_timestamp
Bangalore,India,INVALID_TIMESTAMP,28.2,55.0,14.5,null,Foggy,null,2.8,null,null,null,STN_BLR_02,SkyWatch,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_08.csv,invalid_timestamp
Hyderabad,India,INVALID_TIMESTAMP,35.3,60.0,21.8,null,Thunderstorm,null,1.3,null,null,null,STN_HYD_02,SkyWatch,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_08.csv,invalid_timestamp


In [0]:
for t in [BRONZE_TABLE, STATION_TABLE, SILVER_TABLE, QUARANTINE_TABLE, CONTROL_TABLE]:
    try:
        cnt = spark.table(t).count()
        print(f"OK: {t} (rows={cnt})")
    except Exception as e:
        print(f"MISSING/ERROR: {t} -> {str(e)[:140]}")

OK: weather_project.default.bronze_weather_raw (rows=1620)
OK: weather_project.default.station_master (rows=12)
OK: weather_project.default.silver_weather_clean (rows=1285)
OK: weather_project.default.silver_weather_quarantine (rows=155)
OK: weather_project.default.pipeline_control (rows=1)


In [0]:
# Step 1 — How many rows does CDF give us?
print(f"CDF rows (Bronze): {cdf_df.count()}")

# Step 2 — How many after JOIN?
print(f"Rows after JOIN  : {joined_df.count()}")

# Step 3 — How many clean vs quarantine?
print(f"Clean rows       : {silver_ready_df.count()}")
print(f"Quarantine rows  : {quarantine_df.count()}")
print(f"Total after split: {silver_ready_df.count() + quarantine_df.count()}")

CDF rows (Bronze): 1620
Rows after JOIN  : 1620
Clean rows       : 1285
Quarantine rows  : 155
Total after split: 1440


In [0]:
import json
userdefined function instead of from_json
jaon.loads(json_string)
